In [10]:
# Forward function
from rhythmic_sharing import RhythmicNetwork

# network = RhythmicNetwork(
#     dt=1,
#     input_dims=x.shape[0],
#     spectral_radius=0.7,
#     input_weight=1.0,
#     average_degree_nodes=8,
#     num_nodes=256,
#     leakage=0.5
# )

def rhythmic_forward(network, x, omega, eps=None):
    # print(network, x, omega)
    
    network.delete_history_and_update_natural_frequencies(omega)
    network.train(x)

    if eps is not None:
        network.epsilon1 = eps[0]
        network.epsilon2 = eps[0]
    
    return network.node_states_history[-1], network.get_natural_frequencies()

In [5]:
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
import optax

In [36]:
def init_mlp_params(key, sizes):
    """
    Initializes parameters for a multi-layer perceptron (MLP).
    
    Args:
    key (int): Random seed for parameter initialization.
    sizes (list): List of integers representing the sizes of each layer.
    
    Returns:
    list: A list of dictionaries containing weights ('W') and biases ('b') for each layer.
    """
    key = jax.random.PRNGKey(key)
    keys = jax.random.split(key, len(sizes) - 1)
    params = []
    for k, (in_dim, out_dim) in zip(keys, zip(sizes[:-1], sizes[1:])):
        weight_key, bias_key = jax.random.split(k)
        W = jax.random.normal(weight_key, (in_dim, out_dim)) * jnp.sqrt(2 / in_dim)
        b = jnp.zeros((out_dim,))
        params.append({'W': W, 'b': b})
    return params

@jax.jit
def forward_readout_tanh(X, readout):
    """
    Performs a forward pass through a readout MLP with Tanh activation.
    
    Args:
    X (jax.numpy.ndarray): Input data.
    readout (list): List of layer parameters (weights and biases).
    
    Returns:
    jax.numpy.ndarray: Output of the readout MLP.
    """
    *hidden, last = readout
    for layer in hidden:
        X = jax.nn.tanh(X @ layer['W'] + layer['b'])
    return X @ last['W'] + last['b']


@jax.jit
def init_grad(params):
    """
    Initializes gradients for a parameter tree with zeros.
    
    Args:
    params (dict): Parameter tree.
    
    Returns:
    dict: Gradient tree initialized with zeros.
    """
    return jax.tree.map(lambda p: jnp.zeros(shape=(p.shape)), params)


@jax.jit
def get_perturbations(theta,epsilon,pert_key):
    """
    Generates random perturbations for parameters.
    
    Args:
    theta (dict): Parameter tree.
    epsilon (float): Perturbation magnitude.
    pert_key (int): Random seed for perturbation generation.
    
    Returns:
    dict: Perturbation tree.
    """
    key = jax.random.PRNGKey(pert_key, impl=None)
    return jax.tree.map(
        lambda p: jax.random.choice(key, jnp.array([-1,1])*epsilon, shape=(p.shape)), theta
    )

# @jax.jit
# def collect_grad(perts,delta_c,eta,G):
#     """
#     Updates the gradient estimate using perturbations and cost differences.
    
#     Args:
#     perts (dict): Perturbation tree.
#     delta_c (float): Cost difference.
#     eta (float): Learning rate.
#     G (dict): Current gradient estimate.
    
#     Returns:
#     dict: Updated gradient estimate.
#     """
#     return jax.tree.map(
#         lambda G, p: G + p*delta_c*eta, G, perts
#     )

@jax.jit
def collect_grad(perts, delta_c, eta_dict, G):
    """
    Updates the gradient estimate using perturbations and cost differences.

    Args:
    perts (dict): Perturbation tree.
    delta_c (float): Cost difference (scalar).
    eta_dict (dict): Learning rate per parameter key.
    G (dict): Current gradient estimate.

    Returns:
    dict: Updated gradient estimate.
    """
    return jax.tree_util.tree_map(
        lambda g, p, eta: g + p * delta_c * eta * -1,
        G, perts, eta_dict
    )

@jax.jit
def apply_perturbations(theta,perturbations):
    """
    Applies perturbations to parameters.
    
    Args:
    theta (dict): Parameter tree.
    perturbations (dict): Perturbation tree.
    
    Returns:
    dict: Updated parameter tree with applied perturbations.
    """
    return jax.tree.map(lambda param, pert: param+pert, theta, perturbations)

@jax.jit
def MGD_update(params,G):
    """
    Updates parameters using the gradient estimate.
    
    Args:
    params (dict): Parameter tree.
    G (dict): Gradient estimate.
    
    Returns:
    dict: Updated parameter tree.
    """
    return jax.tree.map(
        lambda p, G: p - G, params, G
    )

@jax.jit
def loss_BCE(logits,labels):
    """
    Computes the binary cross-entropy loss.
    
    Args:
    logits (jax.numpy.ndarray): Predicted logits.
    labels (jax.numpy.ndarray): Ground truth labels.
    
    Returns:
    float: Binary cross-entropy loss.
    """
    return optax.sigmoid_binary_cross_entropy(logits,labels).mean()

@jax.jit
def compute_accuracy_binary(logits,labels):
    """
    Computes binary classification accuracy.
    
    Args:
    logits (jax.numpy.ndarray): Predicted logits.
    labels (jax.numpy.ndarray): Ground truth labels.
    
    Returns:
    float: Binary classification accuracy.
    """
    pred_class = (logits[:,0] > 0.5).astype(jnp.int32)
    true_class = labels.astype(jnp.int32)
    return jnp.mean(pred_class == true_class)
    
def MGD_cost(readout_params, x, y):
    logits = forward_readout_tanh(x, readout_params)
    return loss_BCE(logits,y)

In [32]:
np.random.seed(10)
import matplotlib.pyplot as plt
from tqdm import tqdm

def split_data(X,y_targets):
    split = int(0.8*len(X))
    trainX = X[:split]
    trainy = y_targets[:split]
    testX = X[split:]
    testy = y_targets[split:]
    return trainX, trainy, testX, testy

dataframe = pd.read_csv(
    'http://storage.googleapis.com/download.tensorflow.org/data/ecg.csv',
    header=None
)

raw_data = dataframe.values
dataframe.head()
labels = raw_data[:1000, -1]
data = raw_data[:1000, 0:-1]

shuff = np.arange(len(labels))
np.random.shuffle(shuff)
data = data[shuff]
labels = labels[shuff]

U_train, y0_train, U_test, y0_test = split_data(data,labels)

eta = {'omega': 10_000, 'readout': [{'W': 1, 'b': 1}, {'W': 1, 'b': 1}]}

readout_params = init_mlp_params(0, [100,128,1])

network = RhythmicNetwork(
    dt=1,
    input_dims=1,
    spectral_radius=0.7,
    input_weight=1.0,
    average_degree_nodes=3,
    num_nodes=100,
    leakage=0.5
)
omega = network.get_natural_frequencies()

accs = []
costs = []
epochs = 50

all_params = {'omega':omega, 'readout': readout_params}

# iterate over training epochs
pert_key = 0
for epoch in range(epochs):
    epoch_costs = []

    #iterate over batchs
    for i in range(U_train.shape[0]):
        u = np.expand_dims(U_train[i], axis=0)
        y = y0_train[i]
    
        state, _ = rhythmic_forward(network, u, all_params["omega"])
        
        c0 = MGD_cost(all_params["readout"], state, y)
        
        gradient = init_grad(all_params)
        
        perts = get_perturbations(all_params,1e-6,pert_key)
        pert_key += 1
        all_params_pert = apply_perturbations(all_params,perts)

        state_pert, _ = rhythmic_forward(network, u, all_params_pert['omega'])
        # print(network.node_states[-1])
        # print(all_params["omega"], all_params_pert['omega'])
        
        c1 = MGD_cost(all_params_pert['readout'], state_pert, y)
        
        delta_c = c1 - c0
        # print(delta_c)
        
        gradient = collect_grad(all_params, delta_c, eta, gradient)
        # print(all_params["omega"])
        
        # update parameters
        all_params = MGD_update(all_params, gradient)
        # print(all_params["omega"])
        
        # reset gradient estimations
        gradient = init_grad(all_params)
        
        # record batched training cost
        epoch_costs.append(c0)
        # print(f"{i} - {c0}")
        
    # recored avg training cost
    costs.append(np.mean(epoch_costs))
    print(np.mean(epoch_costs))
plt.plot(costs)
plt.show()
    
    # # test model accuracy on unseen data
    # acc = compute_accuracy_binary(readout_params,s0_test,U_test,y0_test)
    # accs.append(acc)

0 - 0.9750237464904785
1 - 0.788811445236206
2 - 1.0142552852630615
3 - 0.4323310852050781
4 - 0.45797616243362427
5 - 0.6544622182846069
6 - 1.0240707397460938
7 - 0.9899203181266785
8 - 0.979935884475708
9 - 0.9074118137359619
10 - 0.7463600635528564
11 - 1.0824437141418457
12 - 0.4392390251159668
13 - 0.9886539578437805
14 - 1.01906418800354
15 - 1.0894423723220825
16 - 0.7411105036735535
17 - 1.0235620737075806
18 - 0.8404462337493896
19 - 1.0392659902572632
20 - 0.5335043668746948
21 - 0.9800593256950378
22 - 0.9056894779205322
23 - 0.5434595346450806
24 - 0.9722226858139038
25 - 0.7384616136550903
26 - 0.7551962733268738
27 - 0.7728137969970703
28 - 0.45436567068099976
29 - 0.45733582973480225
30 - 0.6973526477813721
31 - 0.9221814870834351
32 - 1.0184274911880493
33 - 1.0153124332427979
34 - 1.0741934776306152
35 - 1.0846261978149414
36 - 1.021451711654663
37 - 0.4503270089626312
38 - 1.031617283821106
39 - 0.4440881013870239
40 - 0.9466038942337036
41 - 0.6261498928070068
42 - 

KeyboardInterrupt: 

In [35]:
readout_params = init_mlp_params(0, [100,128,1])

network = RhythmicNetwork(
    dt=1,
    input_dims=1,
    spectral_radius=0.7,
    input_weight=1.0,
    average_degree_nodes=3,
    num_nodes=100,
    leakage=0.5
)
omega = network.get_natural_frequencies()

accs = []
costs = []
epochs = 50

all_params = {'readout': readout_params}
eta = {'readout': [{'W': 1, 'b': 1}, {'W': 1, 'b': 1}]}

# iterate over training epochs
pert_key = 0
for epoch in range(epochs):
    epoch_costs = []

    #iterate over batchs
    for i in range(U_train.shape[0]):
        u = np.expand_dims(U_train[i], axis=0)
        y = y0_train[i]
    
        state, _ = rhythmic_forward(network, u, omega) # Not trying to learn omega here
        
        c0 = MGD_cost(all_params["readout"], state, y)
        
        gradient = init_grad(all_params)
        
        perts = get_perturbations(all_params,1e-6,pert_key)
        pert_key += 1
        all_params_pert = apply_perturbations(all_params,perts)

        state_pert, _ = rhythmic_forward(network, u, omega)  # Not trying to learn omega here
        # print(network.node_states[-1])
        # print(all_params["omega"], all_params_pert['omega'])
        
        c1 = MGD_cost(all_params_pert['readout'], state_pert, y)
        
        delta_c = c1 - c0
        
        gradient = collect_grad(all_params, delta_c, eta, gradient)
        
        # update parameters
        all_params = MGD_update(all_params, gradient)
        
        # reset gradient estimations
        gradient = init_grad(all_params)
        
        # record batched training cost
        epoch_costs.append(c0)
        # print(f"{i} - {c0}")
        
    # recored avg training cost
    costs.append(np.mean(epoch_costs))
    print(np.mean(epoch_costs))
plt.plot(costs)
plt.show()
    
    # # test model accuracy on unseen data
    # acc = compute_accuracy_binary(readout_params,s0_test,U_test,y0_test)
    # accs.append(acc)

0 - 0.9750237464904785
1 - 0.7877627611160278
2 - 1.0143262147903442
3 - 0.43235844373703003
4 - 0.457160621881485
5 - 0.6596584916114807
6 - 1.0251612663269043
7 - 0.9898070096969604
8 - 0.9792025089263916
9 - 0.9094120264053345
10 - 0.749853789806366
11 - 1.08146071434021
12 - 0.4385058879852295
13 - 0.9909265041351318
14 - 1.016108751296997
15 - 1.0885937213897705
16 - 0.7430021166801453
17 - 1.0232484340667725
18 - 0.8400712013244629
19 - 1.0374350547790527
20 - 0.5354579091072083
21 - 0.981024980545044
22 - 0.910585880279541
23 - 0.5432828664779663
24 - 0.9722776412963867
25 - 0.7428749203681946
26 - 0.7596442699432373
27 - 0.7761863470077515
28 - 0.45444875955581665
29 - 0.4567468762397766
30 - 0.6992775201797485
31 - 0.9224451780319214
32 - 1.0143793821334839
33 - 1.015153169631958
34 - 1.0700557231903076
35 - 1.082192063331604
36 - 1.0147991180419922
37 - 0.4472317695617676
38 - 1.0220701694488525
39 - 0.440951406955719
40 - 0.9478367567062378
41 - 0.635861337184906
42 - 1.0049

KeyboardInterrupt: 

In [38]:
readout_params = init_mlp_params(0, [100,128,1])

network = RhythmicNetwork(
    dt=1,
    input_dims=1,
    spectral_radius=0.7,
    input_weight=1.0,
    average_degree_nodes=3,
    num_nodes=100,
    leakage=0.5
)
omega = network.get_natural_frequencies()
eps = np.asarray([network.epsilon1, network.epsilon2])

accs = []
costs = []
epochs = 50

all_params = {'omega':omega, 'readout': readout_params, "eps": eps}
eta = {'omega': 10_000, 'readout': [{'W': 1, 'b': 1}, {'W': 1, 'b': 1}], 'eps': 1}

# iterate over training epochs
pert_key = 0
for epoch in range(epochs):
    epoch_costs = []

    #iterate over batchs
    for i in range(U_train.shape[0]):
        u = np.expand_dims(U_train[i], axis=0)
        y = y0_train[i]
    
        state, _ = rhythmic_forward(network, u, all_params["omega"], eps=all_params["eps"])
        
        c0 = MGD_cost(all_params["readout"], state, y)
        
        gradient = init_grad(all_params)
        
        perts = get_perturbations(all_params,1e-6,pert_key)
        pert_key += 1
        all_params_pert = apply_perturbations(all_params,perts)

        state_pert, _ = rhythmic_forward(network, u, all_params_pert["omega"], eps=all_params_pert["eps"])
        # print(network.node_states[-1])
        # print(all_params["omega"], all_params_pert['omega'])
        
        c1 = MGD_cost(all_params_pert['readout'], state_pert, y)
        
        delta_c = c1 - c0
        
        gradient = collect_grad(all_params, delta_c, eta, gradient)
        
        # update parameters
        all_params = MGD_update(all_params, gradient)
        
        # reset gradient estimations
        gradient = init_grad(all_params)
        
        # record batched training cost
        epoch_costs.append(c0)
        # print(f"{i} - {c0}")
        
    # recored avg training cost
    costs.append(np.mean(epoch_costs))
    print(np.mean(epoch_costs))
plt.plot(costs)
plt.show()
    
    # # test model accuracy on unseen data
    # acc = compute_accuracy_binary(readout_params,s0_test,U_test,y0_test)
    # accs.append(acc)

0 - 0.9750237464904785
1 - 0.8109003305435181
2 - 1.017359733581543
3 - 0.43022939562797546
4 - 0.44319379329681396
5 - 0.642018735408783
6 - 1.0344945192337036
7 - 1.0178461074829102
8 - 0.9681441783905029
9 - 0.905920147895813
10 - 0.7108080387115479
11 - 1.0918101072311401
12 - 0.4313275218009949
13 - 1.0370206832885742
14 - 1.0218085050582886
15 - 1.1300777196884155
16 - 0.7675508260726929
17 - 1.0417033433914185
18 - 0.8143433928489685
19 - 1.0585968494415283
20 - 0.531723141670227
21 - 1.0271248817443848


KeyboardInterrupt: 